In [ ]:
import pandas as pd


In [51]:
data = pd.read_csv('../data/train/train.csv')
print(data.shape)
data.isnull().sum()

data['CabinKnown'] = data['Cabin'].notna()

data['Age'] = data['Age'].fillna(data['Age'].median())

data['Embarked'] = data['Embarked'].fillna('S')


(891, 12)


In [3]:
data_procced_categories = data.copy()
data_procced_categories['CabinKnown'] = data_procced_categories['CabinKnown'].astype('int32')
replace_dict = {'male': 0, 'female': 1}
data_procced_categories['Sex'] = data_procced_categories['Sex'].replace(replace_dict)

# data_procced_categories['FareLog'] = np.log1p(data_procced_categories['Fare'])

data_procced_categories.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,CabinKnown
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,0


In [8]:
from sklearn.preprocessing import StandardScaler

data_scaled = data_procced_categories.copy()

features_to_scale = ['Age', 'Fare', 'SibSp', 'Parch']
scaler = StandardScaler()
data_scaled[features_to_scale] = scaler.fit_transform(
    data_procced_categories[features_to_scale]
)

data_scaled[features_to_scale].describe()
data_scaled

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,CabinKnown,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,-0.565736,0.432793,-0.473674,A/5 21171,-0.502445,NaN,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0.663861,0.432793,-0.473674,PC 17599,0.786845,C85,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",1,-0.258337,-0.474545,-0.473674,STON/O2. 3101282,-0.488854,NaN,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0.433312,0.432793,-0.473674,113803,0.420730,C123,1,0,1
4,5,0,3,"Allen, Mr. William Henry",0,0.433312,-0.474545,-0.473674,373450,-0.486337,NaN,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",0,-0.181487,-0.474545,-0.473674,211536,-0.386671,NaN,0,0,1
887,888,1,1,"Graham, Miss. Margaret Edith",1,-0.796286,-0.474545,-0.473674,112053,-0.044381,B42,1,0,1
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",1,-0.104637,0.432793,2.008933,W./C. 6607,-0.176263,NaN,0,0,1
889,890,1,1,"Behr, Mr. Karl Howell",0,-0.258337,-0.474545,-0.473674,111369,-0.044381,C148,1,0,0


In [49]:
# data_procced_categories = pd.get_dummies(data_procced_categories, columns=['Embarked'], drop_first=True, dtype=int)
data_scaled = pd.get_dummies(data_procced_categories, columns=['Embarked'], drop_first=True, dtype=int)
data_procced_categories

KeyError: "None of [Index(['Embarked'], dtype='str')] are in the [columns]"

In [50]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

# train_X = data_procced_categories.copy()
train_X = data_scaled.copy()
train_X = train_X.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
train_y = data_procced_categories['Survived']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metric = []
print(train_X)
for train_indices, val_indices in skf.split(train_X, train_y):
    x_train_fold = train_X.iloc[train_indices]
    x_val_fold = train_X.iloc[val_indices]

    y_train_fold = train_y.iloc[train_indices]
    y_val_fold = train_y.iloc[val_indices]

    model = LogisticRegression(fit_intercept=True, max_iter=1000, C=0.45)

    model.fit(x_train_fold, y_train_fold)

    y_pred = model.predict(x_val_fold)

    accuracy = accuracy_score(y_val_fold, y_pred)
    fold_metric.append(accuracy)

print(f"Mean accuracy: {sum(fold_metric) / len(fold_metric):.4f}")
print(fold_metric)
# Mean accuracy: 0.7901 - 5 folds
# Mean accuracy: 0.7968 - 10 folds
# Mean accuracy: 0.7980 - 5 folds - fareLog



     Pclass Sex       Age     SibSp     Parch      Fare  CabinKnown  \
0         3   0 -0.565736  0.432793 -0.473674 -0.502445           0   
1         1   1  0.663861  0.432793 -0.473674  0.786845           1   
2         3   1 -0.258337 -0.474545 -0.473674 -0.488854           0   
3         1   1  0.433312  0.432793 -0.473674  0.420730           1   
4         3   0  0.433312 -0.474545 -0.473674 -0.486337           0   
..      ...  ..       ...       ...       ...       ...         ...   
886       2   0 -0.181487 -0.474545 -0.473674 -0.386671           0   
887       1   1 -0.796286 -0.474545 -0.473674 -0.044381           1   
888       3   1 -0.104637  0.432793  2.008933 -0.176263           0   
889       1   0 -0.258337 -0.474545 -0.473674 -0.044381           1   
890       3   0  0.202762 -0.474545 -0.473674 -0.492378           0   

     Embarked_Q  Embarked_S  
0             0           1  
1             0           0  
2             0           1  
3             0           1